# Block 06: Convergence, Initialization, and Rank Selection

**Goal**  
Quantify rank saturation and EMPCA convergence traces on controlled QPSimulator families.

**Checklist alignment**  
Implements E3 and E9.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [2]:
out = run_block06_convergence_suite(cfg)
display(out["rank_summary_df"])
display(out["convergence_summary_df"])

,setup,k,chi2_proxy_mean,chi2_proxy_std,relative_drop_vs_k1
8,setup_B_multiD,1,45540.166431,10226.902625,0.000000
9,setup_B_multiD,2,37772.825245,3368.700314,0.170560
10,setup_B_multiD,3,35679.996449,1985.301845,0.216516
11,setup_B_multiD,4,35071.854101,1568.780146,0.229870
12,setup_B_multiD,5,34809.160625,1302.479280,0.235638
13,setup_B_multiD,6,34705.808284,1233.030020,0.237908
14,setup_B_multiD,7,34675.486812,1211.462442,0.238574
15,setup_B_multiD,8,34665.793115,1206.328846,0.238786


,k,init,iterations_used,monotone_nonincreasing,iter_to_relative_tol_1e-6,chi2_final
0,1,random,80,True,4,44967.402758
1,1,svd,80,True,1,44967.402758
2,2,random,80,True,6,35885.640659
3,2,svd,80,True,1,35885.640659
4,3,random,80,True,7,33473.940932
5,3,svd,80,True,1,33473.940932


In [3]:
fig, ax = plt.subplots(figsize=(6, 4))
rank_df = out["rank_summary_df"]
ax.plot(rank_df["k"], rank_df["chi2_proxy_mean"], marker="o")
ax.set_xlabel("rank k")
ax.set_ylabel("held-out weighted residual")
ax.set_title("Rank saturation on multi-dimensional family")
save_figure(fig, dirs["figures"] / "block_06_rank_saturation.png")
plt.close(fig)

In [4]:
fig, ax = plt.subplots(figsize=(6, 4))
conv = out["convergence_df"]
for (k, init), group in conv.groupby(["k", "init"]):
    if k in (1, 2):
        ax.plot(group["iteration"], group["chi2"], label=f"k={k}, {init}")
ax.set_xlabel("iteration")
ax.set_ylabel("chi2 surrogate")
ax.set_title("EMPCA convergence traces")
ax.legend()
save_figure(fig, dirs["figures"] / "block_06_convergence.png")
plt.close(fig)

In [5]:
mono_path = dirs["tables"] / "block_06_e3_monotonicity.csv"
conv_path = dirs["tables"] / "block_06_e9_convergence.csv"
rank_path = dirs["tables"] / "block_06_rank_selection.csv"
manifest_path = dirs["manifests"] / "block_06_rank_selection_summary.json"
save_dataframe(out["monotonicity_df"], mono_path)
save_dataframe(out["convergence_df"], conv_path)
save_dataframe(out["rank_summary_df"], rank_path)
save_json({"summary_rows": out["convergence_summary_df"].to_dict(orient="records")}, manifest_path)
pd.DataFrame([manifest_row("block_06", "theorem-support", str(rank_path.relative_to(REPO_ROOT)), cfg)])

,block_id,regime,output_path,seed,trace_len,pretrigger
0,block_06,theorem-support,results/tables/block_06_rank_selection.csv,314159,32768,4000
